# 📦 Citadel Publish Contract - Testing Center

## Overview

Publish and validate **three protected AI assets** through the AI Hub Gateway using the **Citadel Publish Contract**, then confirm reachability, policy application, usage tracking, and resiliency.

| # | Asset | `assetType` | Result |
|---|---|---|---|
| 1 | Weather Tool | `mcp-from-api` | Sample `weather-api` exposed as an MCP tool server |
| 2 | Microsoft Learn Tool | `mcp-existing` | Remote MS Learn MCP server published + protected |
| 3 | HR Chat Agent | `a2a` | Foundry-hosted agent published as a native A2A endpoint |

## Expected outcomes

- Three assets published via Bicep (`az deployment sub create`)
- MCP `initialize` + `tools/list` handshakes succeed through the gateway
- The A2A agent card is retrievable at `/.well-known/agent.json` and a JSON-RPC `message/send` routes to Foundry
- `mcp-usage` / `a2a-usage` custom metrics appear in Application Insights
- Each remote MCP backend has a native circuit breaker attached
- The published **HR agent** answers an HR question via gateway-routed A2A, and a direct MCP `tools/call` returns a result

## Azure Prerequisites

- An existing Citadel Governance Hub deployment (APIM + App Insights + Cosmos + usage Logic Apps)
- The sample APIs deployed (`isMCPSampleDeployed = true`) so `weather-api` exists as the API→MCP source
- A Foundry **prompt agent** to expose via A2A (see `citadel-agent-frameworks-tests.ipynb` to create one)
- Permission to assign roles on the Foundry project (Owner / User Access Administrator) — step 3️⃣.1 grants the APIM identity **Foundry access** automatically
- Azure credentials with permission to deploy at subscription scope

> **Access Contract note:** publishing (phase 1) does not grant access. This notebook then deploys a **real Citadel Access Contract** — a single `MULTI-` product that grants **all three published asset types at once** and applies **asset-type-aware** policies (LLM token limits + model RBAC; request-based rate limits for Tools and Agents). The contract mints the `api-key` used by the validation steps below.

### Shared workflow and safety

Run from the repository root or `validation`. Explicit `overrides` win over the
selected azd environment. Annotate deployed model operations with `capabilities`
(or `VALIDATION_MODEL_CAPABILITIES`); unknown capabilities block required coverage.
Image operations use `image-generation` and `image-edit`; affinity uses `responses`.
Definitions and policy XML remain visible. Preview the generated **`.bicepparam`**
in one cell, optionally edit it, then deploy it in the following cell; the shared
deploy helper compiles the actual parameter file before submission.
Generated artifacts live under ignored `validation/.runs`. Tokens/keys stay in
memory. `run.finish()` raises for required FAIL/BLOCKED/empty runs. Cleanup is off
and can only delete resources absent before this run. Cloud execution has real cost
and side effects; local source validation is not a cloud feature verdict.

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Set `init_from_azd = True` to autoload the governance hub RG/location/subscription from your active `azd` environment, or `False` to fill the `REPLACE` values manually. Provide the Foundry agent coordinates for the A2A asset.

### 0️⃣ Initialize Notebook Variables

Configure this notebook's environment, explicit feature opt-ins, and run-owned resources without deploying anything.

**Input:** The selected repository kernel; edit the configuration values below and provide required secrets through environment variables.

**Output:** Configuration globals for the following initialization step; no cloud deployment or test is performed.


### Automatic models and optional overrides

Leave `environment=None`, azd enabled, `operation_overrides=None`, and
`skipped_models=None` to use the selected deployment without editing model lists.
Operations default to name-based **assumptions**, verified by real requests:
`image`/`flux` → image generation/edit; otherwise `embedding`/`text` → embeddings;
otherwise `gpt` → chat and Responses; other names → chat. Matching ignores case.

Optional examples (replace example names with names in the loaded inventory):

```python
# Replace operations only for these models; other models retain automatic mapping.
operation_overrides = {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
# Use operation_overrides = {} to discard stored azd operation overrides.
# Disable one model's tests with an empty override: {"my-model": []}.
skipped_models = ["mini", "legacy"]  # Case-insensitive substrings; backend deployment is unchanged.
overrides = {"resource_group": "rg-other-hub", "apim_name": "apim-other-hub"}  # Explicit azd-value overrides.
# To require a complete manually mapped catalogue: overrides["infer_operations"] = False.
```

Streaming is assumed for inferred chat models. Explicit mappings need `streaming`
when that scenario is selected. Responses GET/input-items returning HTTP 404 are warnings/SKIP for every model,
including explicit overrides, not successful retrieval. Other unsupported-operation
exceptions remain limited to inferred GPT-mini models; explicit mappings require
those non-404 retrieval checks. Authentication, throttling, and server failures
are never silently accepted.

### Scenario-specific input examples


The A2A scenario publishes an existing Foundry agent. Example target inputs:
`foundry_account_name = "aif-agent-host"; foundry_project_name = "project-agents"; foundry_agent_name = "hr-agent"`.
Account, project and project endpoint use an unambiguous existing azd Foundry target when present. An existing agent name must still be supplied (`VALIDATION_FOUNDRY_AGENT_NAME` / `FOUNDRY_AGENT_NAME` in azd, or the explicit variable); model backend metadata does not identify an agent.
`app_insights_name_override = "appi-my-gateway"` disambiguates telemetry when
needed. `enable_a2a_asset = False` explicitly opts out of A2A.
If `use_access_contract_kv = True`, provide the Key Vault name and, for an external
vault, its subscription and resource group (for example `access_contract_kv_name = "kv-consumers"`).


In [ ]:
from pathlib import Path
import sys
repo_root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if repo_root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(repo_root / "shared"))
import utils
with utils.notebook_step('0️⃣ Initialize Notebook Variables', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='Configuration globals for the following initialization step; no cloud deployment or test is performed.', namespace=globals()):
    import os, sys, json, time, uuid, base64, requests
    from pathlib import Path
    from apimtools import APIMClientTool
    from validationtools import ValidationContext, ValidationRun, RequestTransport, ValidationBlocked
    from contracttools import write_parameters, deploy_contract, access_policy, preview
    from scenariotools import access_parameters, contract_identity, cleanup_products, policy_xml as normalize_policy_xml, completion_text, save_manifest

    init_from_azd = True
    environment = None  # Explicit azd environment, or the active environment.
    overrides = {}  # Explicit values win: resource_group/location/subscription_id/apim_name.
    # Supply capability metadata, not name patterns, e.g.
    # overrides["capabilities"] = {"your-deployed-model": ["chat", "responses"]}
    run_id = uuid.uuid4().hex[:12]
    cleanup_delete_contract = False

    enable_a2a_asset = True
    foundry_account_name = ""
    foundry_project_name = ""
    foundry_agent_name = ""  # Existing agent identity is never invented from model registrations.
    foundry_project_endpoint = ""  # Prefer the actual project endpoint exported by azd.
    use_access_contract_kv = False
    access_contract_kv_name = ""
    access_contract_kv_subscription_id = ""
    access_contract_kv_resource_group = ""
    enable_agent_framework = False  # Real SDK example is retained; opt in with installed dependencies.
    app_insights_name_override = None
    telemetry_poll_attempts = 6
    telemetry_poll_seconds = 10
    enable_circuit_breaker_fault_injection = False  # Configuration inspection is not trip/recovery proof.
    results = {}  # One collector before the first check; never reset downstream.
    publish_deployment_name = "publish-" + run_id
    def _is_unset(value):
        return value is None or value == "" or value == "REPLACE"

    # None uses azd operation overrides when present, otherwise automatic name-based mapping.
    # Example: {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
    # An empty list for one model disables its tests; explicit lists replace that model's inferred operations.
    # A supplied dictionary replaces azd's operation map; {} resets it to automatic mapping.
    operation_overrides = None
    # None uses VALIDATION_SKIPPED_MODELS from azd (default []); matches substrings without case sensitivity.
    # Example: ["mini", "legacy"] skips any model containing either keyword in validation, not deployment.
    skipped_models = None


<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

### 1️⃣ Verify Azure CLI and Connected Subscription

Resolve the selected deployment and validate this scenario's mandatory inputs before deployment or feature tests.

**Input:** Configuration from the previous cell, resource coordinates, inferred or explicitly overridden model operations, and enabled integration prerequisites.

**Output:** Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.


In [ ]:
with utils.notebook_step('1️⃣ Verify Azure CLI and Connected Subscription', inputs={'enable_a2a_asset': globals().get('enable_a2a_asset'), 'use_access_contract_kv': globals().get('use_access_contract_kv'), 'app_insights_name_override': globals().get('app_insights_name_override'), 'foundry_account_name': globals().get('foundry_account_name'), 'foundry_project_name': globals().get('foundry_project_name'), 'foundry_agent_name': globals().get('foundry_agent_name')}, expected='Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.', namespace=globals()):
    run = ValidationRun("publish-contract")
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=init_from_azd, operation_overrides=operation_overrides, skipped_models=skipped_models)
    mandatory_inputs = {
        "gateway.subscription_id": context.config.get("subscription_id"),
        "gateway.resource_group": context.config.get("resource_group"),
        "gateway.location": context.config.get("location"),
        "gateway.apim_name": context.config.get("apim_name"),
    }
    foundry_account_name = foundry_account_name or (context.config.get("foundry_account_name") or context.config.get("FOUNDRY_ACCOUNT_NAME", ""))
    foundry_project_name = foundry_project_name or (context.config.get("foundry_project_name") or context.config.get("FOUNDRY_PROJECT_NAME", ""))
    foundry_agent_name = foundry_agent_name or (context.config.get("foundry_agent_name") or context.config.get("VALIDATION_FOUNDRY_AGENT_NAME") or context.config.get("FOUNDRY_AGENT_NAME", ""))
    foundry_project_endpoint = foundry_project_endpoint or (context.config.get("foundry_project_endpoint") or context.config.get("FOUNDRY_PROJECT_ENDPOINT", ""))
    if enable_a2a_asset and not (foundry_account_name and foundry_project_name and foundry_project_endpoint):
        # AI_FOUNDRY_SERVICES exports actual project endpoints, not agent identities.
        candidates = []
        for service in context.config.get("ai_foundry_services", []):
            endpoint = service.get("foundryProjectEndpoint", "")
            account_name = service.get("cognitiveServiceName") or service.get("accountName") or service.get("name")
            if not endpoint or "/api/projects/" not in endpoint or not account_name:
                continue
            project_name = endpoint.rstrip("/").rsplit("/api/projects/", 1)[1]
            if foundry_account_name and foundry_account_name != account_name:
                continue
            if foundry_project_name and foundry_project_name != project_name:
                continue
            candidates.append((account_name, project_name, endpoint))
        candidates = list(dict.fromkeys(candidates))
        if len(candidates) == 1:
            account_name, project_name, endpoint = candidates[0]
            foundry_account_name = foundry_account_name or account_name
            foundry_project_name = foundry_project_name or project_name
            foundry_project_endpoint = foundry_project_endpoint or endpoint
    if enable_a2a_asset:
        mandatory_inputs.update({
            "a2a.foundry_account_name": foundry_account_name,
            "a2a.foundry_project_name": foundry_project_name,
            "a2a.foundry_agent_name (existing agent; VALIDATION_FOUNDRY_AGENT_NAME)": foundry_agent_name,
        })
    if use_access_contract_kv:
        access_contract_kv_name = access_contract_kv_name or context.config.get("key_vault_name", "")
        access_contract_kv_subscription_id = access_contract_kv_subscription_id or context.config.get("subscription_id")
        access_contract_kv_resource_group = access_contract_kv_resource_group or context.config.get("resource_group")
        mandatory_inputs.update({
            "keyvault.name": access_contract_kv_name,
            "keyvault.subscription_id": access_contract_kv_subscription_id,
            "keyvault.resource_group": access_contract_kv_resource_group,
        })
    utils.require_inputs(mandatory_inputs)
    app_insights_name_override = app_insights_name_override or (context.config.get("apim_app_insights_name") or context.config.get("APIM_APP_INSIGHTS_NAME"))
    app_insights_resource_group = context.config["resource_group"]
    app_insights_subscription_id = context.config["subscription_id"]
    if not app_insights_name_override:
        logger_query = utils.run(
            f"az apim logger list --resource-group {context.config['resource_group']} "
            f"--service-name {context.config['apim_name']} "
            "--query \"[?loggerType=='applicationInsights'].resourceId\" -o json")
        component_ids = list(dict.fromkeys(item for item in (logger_query.json_data or [])
                                          if isinstance(item, str) and item)) if logger_query.success else []
        if len(component_ids) == 1:
            parts = component_ids[0].strip("/").split("/")
            if len(parts) == 8 and [part.casefold() for part in parts[4:7]] == ["providers", "microsoft.insights", "components"]:
                app_insights_name_override = parts[7]
                app_insights_subscription_id, app_insights_resource_group = parts[1], parts[3]
    utils.require_inputs({"publication.app_insights_name (APIM_APP_INSIGHTS_NAME or exact APIM logger resourceId)":
                          app_insights_name_override})
    utils.require_capabilities(context.catalog, context.catalog.models_for("chat"))
    governance_hub_resource_group = context.config["resource_group"]
    location = context.config["location"]
    subscription_id = context.config["subscription_id"]
    llm_backends_config = context.config["llm_backends"]
    run_dir = repo_root / "validation" / ".runs" / "publish-contract" / run_id
    transport = RequestTransport(timeout=120)
    utils.format_output('1️⃣ Verify Azure CLI and Connected Subscription', {'value_1': {"environment": environment or "active", "resource_group": governance_hub_resource_group,
           "location": location, "subscription_id": subscription_id}}, status='info', part='RESULT')
    utils.format_output('1️⃣ Verify Azure CLI and Connected Subscription', {'value_1': context.catalog.report()}, status='info', part='RESULT')

    run_dir.mkdir(parents=True, exist_ok=True)


<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool

Discover the deployed APIM instance and its gateway URL.

### 2️⃣ Initialize APIM Client Tool

Prepare the 2️⃣ Initialize APIM Client Tool state shown below for its subsequent feature-specific requests and assertions.

**Input:** `governance_hub_resource_group`

**Output:** `apimClientTool`, `gateway_url`, `azure_endpoint`, `apim_resource_gateway_url`, `apim_name`, `preexisting_products` are available to the following cells.


In [ ]:
with utils.notebook_step('2️⃣ Initialize APIM Client Tool', inputs={'governance_hub_resource_group': globals().get('governance_hub_resource_group')}, expected='`apimClientTool`, `gateway_url`, `azure_endpoint`, `apim_resource_gateway_url`, `apim_name`, `preexisting_products` are available to the following cells.', namespace=globals()):
    apimClientTool = APIMClientTool(governance_hub_resource_group, apim_resource_name=context.config["apim_name"])
    apimClientTool.initialize_context(context.account, enumerate_subscriptions=False)

    gateway_url = str(apimClientTool.apim_resource_gateway_url).rstrip("/")
    azure_endpoint = gateway_url + "/"
    apim_resource_gateway_url = gateway_url
    apim_name = apimClientTool.apim_resource_name
    preexisting_products = {p.name for p in apimClientTool.client.product.list_by_service(
        governance_hub_resource_group, apim_name)}
    owned_products = set()

    client = apimClientTool.client
    rg, svc = governance_hub_resource_group, apim_name
    preexisting_apis = {a.name for a in client.api.list_by_service(rg, svc)}
    preexisting_backends = {b.name for b in client.backend.list_by_service(rg, svc)}
    owned_asset_apis = set()
    owned_asset_backends = set()
    run_started_at = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


<a id='3'></a>
### 3️⃣ Enable incoming A2A on the Foundry agent

Foundry prompt agents support the responses protocol, but the **A2A endpoint must be activated** with a `PATCH` that sets the agent card and enables the `a2a` protocol. This mirrors [`local/contracts/a2a-foundry-sample-agent.ps1`](../local/contracts/a2a-foundry-sample-agent.ps1) and the Learn guide: [Enable incoming A2A on a Foundry agent](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/enable-agent-to-agent-endpoint).

> Requires the **Foundry User** role on the project. Skipped automatically when `enable_a2a_asset = False`.

### 3️⃣ Enable incoming A2A on the Foundry agent

Run the actual 3️⃣ Enable incoming A2A on the Foundry agent SDK example against the initialized contract and retain its returned content/session evidence.

**Input:** `enable_a2a_asset`, `foundry_account_name`, `foundry_project_name`, `foundry_agent_name`

**Output:** SDK responses, session/agent state, and recorded feature validation; optional disabled paths remain explicit.


In [ ]:
with utils.notebook_step('3️⃣ Enable incoming A2A on the Foundry agent', inputs={'enable_a2a_asset': globals().get('enable_a2a_asset'), 'foundry_account_name': globals().get('foundry_account_name'), 'foundry_project_name': globals().get('foundry_project_name'), 'foundry_agent_name': globals().get('foundry_agent_name')}, expected='SDK responses, session/agent state, and recorded feature validation; optional disabled paths remain explicit.', namespace=globals()):
    a2a_card_backend_url = ""
    a2a_jsonrpc_backend_url = ""
    if enable_a2a_asset and not (_is_unset(foundry_account_name) or _is_unset(foundry_project_name) or _is_unset(foundry_agent_name)):
        base_url = (foundry_project_endpoint or
                    f"https://{foundry_account_name}.services.ai.azure.com/api/projects/{foundry_project_name}").rstrip("/")
        a2a_card_backend_url = f"{base_url}/agents/{foundry_agent_name}/endpoint/protocols/a2a/agentCard/v1.0"
        a2a_jsonrpc_backend_url = f"{base_url}/agents/{foundry_agent_name}/endpoint/protocols/a2a"
        token = utils.run("az account get-access-token --resource https://ai.azure.com --query accessToken -o tsv").text.strip()
        body = {
            "agent_card": {
                "description": "HR Chat Agent published via the AI Hub Gateway (A2A).",
                "version": "1.0",
                "skills": [{"id": "general-qa", "name": "General Q&A", "description": "Answers HR policy questions"}]
            },
            "agent_endpoint": {"protocol_configuration": {"responses": {}, "a2a": {}}}
        }
        r = utils.http_call(requests.patch, f"{base_url}/agents/{foundry_agent_name}?api-version=v1",
                           headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
                           data=json.dumps(body), timeout=60)
        if r.status_code < 300:
            utils.print_ok(f"Incoming A2A enabled on {foundry_agent_name}")
            utils.print_info(f"Agent card backend: {a2a_card_backend_url}")
        else:
            utils.print_error(f"A2A enable failed ({r.status_code}): {r.text[:400]}")
    else:
        utils.print_warning("A2A asset disabled or Foundry agent not fully specified; skipping A2A enablement.")


<a id='3-1'></a>
### 3️⃣.1 Grant the APIM managed identity access on the Foundry project

The published A2A backend authenticates to Foundry with **APIM's user-assigned managed identity** (audience `https://ai.azure.com`) — no keys. For that to work the identity needs a data-plane role on the Foundry project.

This step:
1. Discovers the APIM identity (prefers a **user-assigned** identity; falls back to system-assigned).
2. Grants it the Foundry role on the target project scope.
3. Captures the UAMI **`clientId`**, which is passed to the deployment as `managedIdentityClientId` so the backend embeds MI auth using that specific identity.

> Default role is `Azure AI User`. For least privilege you can switch `foundry_role` to `Foundry Agent Consumer`. Requires Owner / User Access Administrator on the project. Skipped when `enable_a2a_asset = False`.

### 3️⃣.1 Grant the APIM managed identity access on the Foundry project

Prepare the 3️⃣.1 Grant the APIM managed identity access on the Foundry project state shown below for its subsequent feature-specific requests and assertions.

**Input:** `enable_a2a_asset`, `foundry_account_name`, `governance_hub_resource_group`, `apim_name`, `foundry_project_name`

**Output:** `apim_mi_client_id`, `apim_mi_principal_id`, `foundry_role`, `idj`, `ident`, `uami` are available to the following cells.


In [ ]:
with utils.notebook_step('3️⃣.1 Grant the APIM managed identity access on the Foundry project', inputs={'enable_a2a_asset': globals().get('enable_a2a_asset'), 'foundry_account_name': globals().get('foundry_account_name'), 'governance_hub_resource_group': globals().get('governance_hub_resource_group'), 'apim_name': globals().get('apim_name'), 'foundry_project_name': globals().get('foundry_project_name')}, expected='`apim_mi_client_id`, `apim_mi_principal_id`, `foundry_role`, `idj`, `ident`, `uami` are available to the following cells.', namespace=globals()):
    # Identity passed to the deployment ('' => APIM system-assigned identity)
    apim_mi_client_id = ""
    apim_mi_principal_id = ""
    foundry_role = "Foundry Agent Consumer"   # least-privilege alternative: "Foundry Agent Consumer"

    if enable_a2a_asset and not _is_unset(foundry_account_name):
        # 1) APIM identity — prefer a user-assigned identity, else system-assigned
        idj = utils.run(f"az apim show -g {governance_hub_resource_group} -n {apim_name} --query identity -o json",
                        "Retrieved APIM identity", "Failed to read APIM identity")
        if idj.success and idj.json_data:
            ident = idj.json_data
            uami = ident.get("userAssignedIdentities") or {}
            if uami:
                _rid, _val = next(iter(uami.items()))
                apim_mi_client_id = _val.get("clientId", "")
                apim_mi_principal_id = _val.get("principalId", "")
                utils.print_info(f"APIM user-assigned identity: clientId={apim_mi_client_id}")
            elif ident.get("principalId"):
                apim_mi_principal_id = ident["principalId"]
                utils.print_info("APIM system-assigned identity (managedIdentityClientId left empty)")

        # 2) Foundry project resource id (account may live in a different resource group)
        foundry_project_id = ""
        acc = utils.run(f"az cognitiveservices account list --query \"[?name=='{foundry_account_name}'].id\" -o tsv")
        account_id = acc.text.strip() if acc.success else ""
        if account_id:
            foundry_project_id = f"{account_id}/projects/{foundry_project_name}"

        # 3) Grant the role so APIM's identity can call the Foundry agent
        if apim_mi_principal_id and foundry_project_id:
            ra = utils.run(
                f"az role assignment create --assignee-object-id {apim_mi_principal_id} "
                f"--assignee-principal-type ServicePrincipal --role \"{foundry_role}\" --scope {foundry_project_id}",
                f"Granted '{foundry_role}' to APIM identity on the Foundry project",
                "Role assignment failed (may already exist, or you lack Owner/User Access Administrator)")
            if not ra.success:
                utils.print_warning("If the role name is unavailable in your tenant, try 'Foundry Agent Consumer' or assign it in the portal.")
        else:
            utils.print_warning("Could not resolve APIM principalId or Foundry project id; grant the Foundry role manually.")
    else:
        utils.print_info("A2A asset disabled; skipping Foundry access grant.")


<a id='3-2'></a>
### 3️⃣.2 Ensure the sample source API exists (`weather-api`) — subscription-protected

The Weather Tool is published with `assetType = mcp-from-api`, which turns an **existing** APIM API into an MCP server. That source API (`weather-api`) is part of the main accelerator but is only deployed when `isMCPSampleDeployed = true`, so it may be missing on your gateway — which makes the publish contract deployment fail.

This step creates `weather-api` on demand (idempotent) from the same spec and mock policy shipped in the accelerator ([`openapi.json`](../bicep/infra/modules/apim/sample/weather/openapi.json) + [`policy.xml`](../bicep/infra/modules/apim/sample/weather/policy.xml)), including its `get-weather` operation that the publish contract references.

> 🔐 **Secure source API (key forwarding).** The source API is created **subscription-protected** and reads its key from a **custom header** (`x-mcp-sub-key`). An `mcp-from-api` server forwards `tools/call` to the source API *internally*; APIM strips the standard subscription headers (`api-key` / `Ocp-Apim-Subscription-Key`) before that hop, but leaves a non-standard header intact. The Weather Tool is then published with `forwardSubscriptionKeyToSource: true` (step 4️⃣), so the MCP policy injects the caller's contract key into `x-mcp-sub-key`. Result: the **same contract key** reaches both the MCP tool and the raw API, and **direct calls without a key return `401`** (no anonymous access). The access contract (step 5️⃣) adds `weather-api` to the product so that one key authorizes both.


### 3️⃣.2 Ensure the sample source API exists (weather-api) — subscription-protected

Exercise 3️⃣.2 Ensure the sample source API exists (weather-api) — subscription-protected and compare the actual response/evidence with the expected feature behavior.

**Input:** `governance_hub_resource_group`, `apim_name`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('3️⃣.2 Ensure the sample source API exists (weather-api) — subscription-protected', inputs={'governance_hub_resource_group': globals().get('governance_hub_resource_group'), 'apim_name': globals().get('apim_name')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    # Ensure the sample 'weather-api' exists — it is the source API for the mcp-from-api Weather Tool.
    # The main accelerator only deploys it when isMCPSampleDeployed=true, so create it on demand here.
    from azure.mgmt.apimanagement.models import (
        ApiCreateOrUpdateParameter, PolicyContract, SubscriptionKeyParameterNamesContract)

    _client = apimClientTool.client
    _rg, _svc = governance_hub_resource_group, apim_name
    weather_api_id = "weather-api"
    # Custom (non-standard) header the source API reads its key from. Standard subscription headers
    # (api-key / Ocp-Apim-Subscription-Key) are stripped by APIM before the internal tools/call hop,
    # so a custom name is required for the forwarded key to survive. Reused by steps 4️⃣ + 5️⃣.
    weather_source_key_header = "x-mcp-sub-key"

    _spec_path = str(repo_root / "bicep" / "infra" / "modules" / "apim" / "sample" / "weather" / "openapi.json")
    _policy_path = str(repo_root / "bicep" / "infra" / "modules" / "apim" / "sample" / "weather" / "policy.xml")
    with open(_spec_path, "r", encoding="utf-8") as f:
        _spec = f.read()
    with open(_policy_path, "r", encoding="utf-8") as f:
        _policy = f.read()

    # Upsert (idempotent). SECURE approach: keep the source API subscription-PROTECTED but read the key
    # from the custom header the MCP policy forwards, so there is no anonymous direct access to /weather.
    utils.print_info(f"Ensuring source API '{weather_api_id}' (subscription-protected, custom header {weather_source_key_header})...")
    _client.api.begin_create_or_update(_rg, _svc, weather_api_id, ApiCreateOrUpdateParameter(
        path="weather",
        display_name="Weather API",
        description="Weather API for getting dynamic weather information for a given location.",
        format="openapi+json",
        value=_spec,
        protocols=["https"],
        subscription_required=True,
        subscription_key_parameter_names=SubscriptionKeyParameterNamesContract(
            header=weather_source_key_header, query=weather_source_key_header),
        service_url="https://to-be-replaced-by-policy",
    )).result()
    # Mock response policy — returns synthetic weather data, so no real backend is called.
    _client.api_policy.create_or_update(_rg, _svc, weather_api_id, "policy",
                                        PolicyContract(value=_policy, format="rawxml"))
    utils.print_ok(f"Source API '{weather_api_id}' ready (subscription required, key header {weather_source_key_header}).")

    # Confirm the operation the publish contract references is present.
    _ops = [o.name for o in _client.api_operation.list_by_api(_rg, _svc, weather_api_id)]
    (utils.print_ok if "get-weather" in _ops else utils.print_error)(f"weather-api operations: {_ops}")

    run.check("weather-source-operation", "get-weather" in _ops, expected="get-weather", actual=_ops)


<a id='4'></a>
### 4️⃣ Publish the assets (deploy the Publish Contract)

Generate a `.bicepparam` under the ignored run folder `validation/.runs/publish-contract/<run-id>/publish/` and deploy the `citadel-publish-contracts` module at subscription scope.

### 4️⃣ Publish the assets (deploy the Publish Contract)

Generate the editable parameter artifact for 4️⃣ Publish the assets (deploy the Publish Contract) and display the proposed configuration without deploying it.

**Input:** `enable_a2a_asset`, `a2a_card_backend_url`, `apim_mi_client_id`, `subscription_id`, `governance_hub_resource_group`, `apim_name`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [ ]:
with utils.notebook_step('4️⃣ Publish the assets (deploy the Publish Contract)', inputs={'enable_a2a_asset': globals().get('enable_a2a_asset'), 'a2a_card_backend_url': globals().get('a2a_card_backend_url'), 'apim_mi_client_id': globals().get('apim_mi_client_id'), 'subscription_id': globals().get('subscription_id'), 'governance_hub_resource_group': globals().get('governance_hub_resource_group'), 'apim_name': globals().get('apim_name')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    assets = [
        {
            "assetType": "mcp-from-api", "name": "weather-tool", "displayName": "Weather Tool (MCP)",
            "description": "Weather data operations, published as an MCP tool server.", "path": "weather-tool-mcp",
            "metadata": {"version": "1.0.0", "owner": "Platform Engineering", "classification": "internal"},
            "sourceApiName": "weather-api", "operationNames": ["get-weather"],
            # Secure: forward the caller's contract key to the subscription-protected source API (step 3️⃣.2)
            # via a custom header APIM won't strip, so the same key reaches the tool AND the raw API.
            "forwardSubscriptionKeyToSource": True,
            "sourceSubscriptionKeyHeaderName": globals().get("weather_source_key_header", "x-mcp-sub-key"),
            "publishToApiCenter": False,
        },
        {
            "assetType": "mcp-existing", "name": "ms-learn-tool", "displayName": "Microsoft Learn Tool (MCP)",
            "description": "Microsoft Learn MCP server published through the gateway.", "path": "ms-learn-tool-mcp",
            "transportType": "streamable", "subscriptionRequired": True,
            "metadata": {"version": "1.0.0", "owner": "Knowledge Mgmt", "classification": "public"},
            "backend": {"url": "https://learn.microsoft.com/api/mcp", "authType": "none"}, "publishToApiCenter": False,
        },
    ]
    if enable_a2a_asset and a2a_card_backend_url:
        assets.append({
            "assetType": "a2a", "name": "hr-chat-agent", "displayName": "HR Chat Agent (A2A)",
            "description": "HR assistant published via A2A.", "path": "hr-chat-agent",
            "agentId": foundry_agent_name, "subscriptionRequired": True, "subscriptionKeyHeaderName": "api-key", "agentCardPath": "/.well-known/agent.json",
            "agentCardBackendUrl": a2a_card_backend_url, "jsonRpcPath": "/",
            "metadata": {"version": "1.0.0", "owner": "HR Digital", "classification": "confidential"},
            "backend": {"url": a2a_jsonrpc_backend_url, "authType": "managed-identity", "authConfig": {"resource": "https://ai.azure.com"}},
            "publishToApiCenter": False,
        })


    publish_parameters = {
        "apim": {"subscriptionId": subscription_id, "resourceGroupName": governance_hub_resource_group, "name": apim_name},
        "managedIdentityClientId": apim_mi_client_id,
        "configureCircuitBreaker": True,
        "useAssetTypePathPrefix": True,
        "publishAssets": assets,
    }
    param_path = write_parameters(repo_root / "bicep" / "infra" / "citadel-publish-contracts" / "main.bicep",
        publish_parameters, run_dir / "publish")
    utils.format_output('4️⃣ Publish the assets (deploy the Publish Contract)', {'value_1': preview(param_path)}, status='info', part='RESULT')


### 4️⃣ Publish the assets (deploy the Publish Contract) — continuation 8

Deploy the reviewed artifact for 4️⃣ Publish the assets (deploy the Publish Contract) — continuation 8, then inspect its real resource outputs and ownership.

**Input:** `assets`, `param_path`, `location`, `publish_deployment_name`, `preexisting_apis`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('4️⃣ Publish the assets (deploy the Publish Contract) — continuation 8', inputs={'assets': globals().get('assets'), 'param_path': globals().get('param_path'), 'location': globals().get('location'), 'publish_deployment_name': globals().get('publish_deployment_name'), 'preexisting_apis': globals().get('preexisting_apis')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    publish_outputs = deploy_contract(param_path, location, publish_deployment_name,
        required_outputs=("publishedAssets",))
    published_assets = publish_outputs["publishedAssets"]["value"]
    published_by_name = {asset["name"]: asset for asset in published_assets}
    asset_metric_names = [client.api.get(rg, svc, asset["name"]).display_name for asset in published_assets]
    run.check("published-inventory", {a["name"] for a in assets} == set(published_by_name),
        expected=[a["name"] for a in assets], actual=list(published_by_name))
    for asset in assets:
        if asset["name"] not in preexisting_apis:
            owned_asset_apis.add(asset["name"])
        backend_id = asset["name"] + "-backend"
        if asset["assetType"] in ("mcp-existing", "a2a") and backend_id not in preexisting_backends:
            owned_asset_backends.add(backend_id)


<a id='5'></a>
### 5️⃣ Grant access with a real Access Contract (mixed asset types)

Instead of a throwaway product, deploy a **real Citadel Access Contract** that grants this consumer access to **all three published asset types at once** — the LLM inference APIs, the published **Tools (MCP)**, and the published **Agent (A2A)** — under a **single `MULTI-` product**. The contract is written to `citadel-access-contracts/contracts/<biz>-<usecase>/<env>/`, following the same source-control layout.

The contract's product policy classifies each request with the `set-asset-kind` fragment and applies **asset-type-specific controls**:

| Asset kind | Controls |
| --- | --- |
| **LLM** | model RBAC (`validate-model-access`) + `llm-token-limit` (TPM + quota) |
| **Tool (MCP)** | request-based `rate-limit-by-key` (calls/min) + `quota-by-key` |
| **Agent (A2A)** | request-based `rate-limit-by-key` (calls/min) + `quota-by-key` |

The deployment mints the subscription **api-key** used by the validation steps below. Backward compatible: existing `LLM-` contracts are untouched.

> 🔐 **Optional Key Vault (set `use_access_contract_kv = True` in step 0️⃣):** the contract shares **one** api-key across every asset and publishes it plus **one endpoint secret per asset** to Key Vault. Every endpoint uses the **default auto-generated secret name** (`<code>-<bu>-<useCase>-<env>-<apiName>-endpoint`) so distinct LLM front-doors never collide — e.g. `universal-llm-api` resolves to `/models` and `azure-openai-api` to `/openai` in their own secrets. `foundryApiName` marks the LLM endpoint (`universal-llm-api`, `/models`) for Foundry connections.
>
> **External Key Vault:** set `access_contract_kv_subscription_id` / `access_contract_kv_resource_group` in step 0️⃣ to publish to a Key Vault in a **different subscription / resource group** (the deploying identity needs *Key Vault Secrets Officer* there). Leave both empty to use the hub's Key Vault. Step 5️⃣.1 verifies the secrets landed.


### 5️⃣ Grant access with a real Access Contract (mixed asset types)

Generate the editable parameter artifact for 5️⃣ Grant access with a real Access Contract (mixed asset types) and display the proposed configuration without deploying it.

**Input:** `governance_hub_resource_group`, `apim_name`, `access_contract_kv_subscription_id`, `subscription_id`, `access_contract_kv_resource_group`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [ ]:
with utils.notebook_step('5️⃣ Grant access with a real Access Contract (mixed asset types)', inputs={'governance_hub_resource_group': globals().get('governance_hub_resource_group'), 'apim_name': globals().get('apim_name'), 'access_contract_kv_subscription_id': globals().get('access_contract_kv_subscription_id'), 'subscription_id': globals().get('subscription_id'), 'access_contract_kv_resource_group': globals().get('access_contract_kv_resource_group')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    import os, json, time

    # Real Access Contract that GRANTS access to the published assets (replaces the old throwaway product).
    client = apimClientTool.client
    rg, svc = governance_hub_resource_group, apim_name

    # --- Classify the published assets + discover which LLM inference APIs exist on this gateway ---
    tool_apis  = [a["name"] for a in assets if a["assetType"] in ("mcp-from-api", "mcp-existing")]
    agent_apis = [a["name"] for a in assets if a["assetType"] == "a2a"]
    _candidate_llm = ["universal-llm-api", "azure-openai-api", "unified-ai-api"]
    existing_apis = {a.name for a in client.api.list_by_service(rg, svc)}
    llm_apis = [n for n in _candidate_llm if n in existing_apis]
    granted_apis = llm_apis + tool_apis + agent_apis
    # Source APIs behind forwarding mcp-from-api tools must join the SAME product so the shared key
    # authorizes both the MCP tool and the protected raw API (forwardSubscriptionKeyToSource).
    source_apis = list(dict.fromkeys(
        a["sourceApiName"] for a in assets
        if a["assetType"] == "mcp-from-api" and a.get("forwardSubscriptionKeyToSource") and a.get("sourceApiName") in existing_apis))
    product_apis = granted_apis + [n for n in source_apis if n not in granted_apis]
    utils.print_info(f"Contract grants -> LLM:{llm_apis}  Tools:{tool_apis}  Agents:{agent_apis}  Sources:{source_apis}")

    # --- Contract identity: MULTI- prefix because it mixes asset types (single-type would be LLM-/TOOL-/AGENT-) ---
    biz_unit, use_case_name, env = "Governance", "PublishedAssets-" + run_id, "DEV"
    _types_present = (1 if llm_apis else 0) + (1 if tool_apis else 0) + (1 if agent_apis else 0)
    contract_code = "MULTI" if _types_present > 1 else ("AGENT" if agent_apis else "TOOL" if tool_apis else "LLM")
    product_id = f"{contract_code}-{biz_unit}-{use_case_name}-{env}"
    sub_name = f"{product_id}-SUB-01"
    test_product_id, sub_id = product_id, sub_name  # reused by the cleanup cell

    # --- Per-asset-type limits (kept low for Tools/Agents so the burst tests below trip a 429) ---
    TOOL_CALLS_PER_MIN, AGENT_CALLS_PER_MIN = 20, 10
    allowed_models_csv = ",".join(context.catalog.models_for("chat"))
    if llm_apis and not allowed_models_csv:
        raise ValidationBlocked("Annotate chat capabilities for the LLM grant in this mixed contract.")

    # --- Generate the asset-type-aware product policy (conditional on set-asset-kind) ---
    policy_xml = f'''<policies>
        <inbound>
            <base />
            <!-- COMMON POLICIES (asset-agnostic): add opt-in content safety / custom alerting here. -->
            <set-variable name="contractToolApis" value="{",".join(tool_apis)}" />
            <set-variable name="contractAgentApis" value="{",".join(agent_apis)}" />
            <include-fragment fragment-id="set-asset-kind" />
            <choose>
                <when condition="@(context.Variables.GetValueOrDefault<string>("assetKind","llm") == "llm")">
                    <include-fragment fragment-id="set-llm-requested-model" />
                    <set-variable name="allowedModels" value="{allowed_models_csv}" />
                    <include-fragment fragment-id="validate-model-access" />
                    <llm-token-limit counter-key="@(context.Subscription.Id)" tokens-per-minute="10000" estimate-prompt-tokens="false" token-quota="1000000" token-quota-period="Monthly" />
                    <set-variable name="enableResponseHeaders" value="@(true)" />
                </when>
                <when condition="@(context.Variables.GetValueOrDefault<string>("assetKind","") == "tool")">
                    <rate-limit-by-key calls="{TOOL_CALLS_PER_MIN}" renewal-period="60" counter-key="@(context.Subscription.Id + ":tool")" />
                    <quota-by-key calls="100000" renewal-period="2592000" counter-key="@(context.Subscription.Id + ":tool")" />
                </when>
                <when condition="@(context.Variables.GetValueOrDefault<string>("assetKind","") == "agent")">
                    <rate-limit-by-key calls="{AGENT_CALLS_PER_MIN}" renewal-period="60" counter-key="@(context.Subscription.Id + ":agent")" />
                    <quota-by-key calls="50000" renewal-period="2592000" counter-key="@(context.Subscription.Id + ":agent")" />
                </when>
            </choose>
        </inbound>
        <backend><base /></backend>
        <outbound><base /></outbound>
        <on-error><base /></on-error>
    </policies>'''


    policy_xml = normalize_policy_xml(policy_xml)

    kv_sub_param = access_contract_kv_subscription_id or subscription_id
    kv_rg_param = access_contract_kv_resource_group or rg
    asset_endpoints = [{"apiName": name} for name in granted_apis]
    foundry_api_name = "universal-llm-api" if "universal-llm-api" in llm_apis else granted_apis[0]
    publish_access_contract = {
        "name": "publish-access-" + run_id, "business_unit": biz_unit, "use_case_name": use_case_name,
        "environment": env, "use_keyvault": use_access_contract_kv,
        "apikey_secret": "PUBLISHED-ASSETS-KEY-" + run_id,
        "description": "Mixed asset-type validation access contract",
    }

    publish_access_parameters = access_parameters(context, apimClientTool, publish_access_contract, policy_xml,
        key_vault={"subscriptionId": kv_sub_param, "resourceGroupName": kv_rg_param, "name": access_contract_kv_name} if use_access_contract_kv else None, code=contract_code, api_names=product_apis, service_options={"assetEndpoints": asset_endpoints, "foundryApiName": foundry_api_name})

    publish_access_params_file = write_parameters(
        repo_root / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep",
        publish_access_parameters, run_dir / "publish_access")
    utils.format_output('5️⃣ Grant access with a real Access Contract (mixed asset types)', {'value_1': preview(publish_access_params_file)}, status='info', part='RESULT')


### 5️⃣ Grant access with a real Access Contract (mixed asset types) — continuation 10

Deploy the reviewed artifact for 5️⃣ Grant access with a real Access Contract (mixed asset types) — continuation 10, then inspect its real resource outputs and ownership.

**Input:** `publish_access_params_file`, `location`, `preexisting_products`, `owned_products`, `sub_name`, `publish_access_contract`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('5️⃣ Grant access with a real Access Contract (mixed asset types) — continuation 10', inputs={'publish_access_params_file': globals().get('publish_access_params_file'), 'location': globals().get('location'), 'preexisting_products': globals().get('preexisting_products'), 'owned_products': globals().get('owned_products'), 'sub_name': globals().get('sub_name'), 'publish_access_contract': globals().get('publish_access_contract')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    publish_access_outputs = deploy_contract(publish_access_params_file, location, publish_access_contract["name"],
        required_outputs=("products", "subscriptions"))
    publish_access_product_id, publish_access_subscription_name = contract_identity(
        publish_access_outputs, preexisting_products, owned_products)

    product_id = publish_access_product_id
    sub_name = publish_access_subscription_name
    api_key = apimClientTool.get_subscription_key(sub_name)
    subscriptions = publish_access_outputs["subscriptions"]["value"]
    access_contract_key_secret_name = subscriptions[0].get("keyVaultApiKeySecretName")
    access_contract_endpoint_secret_names = subscriptions[0].get("keyVaultEndpointSecretNames", [])


### 5️⃣ Grant access with a real Access Contract (mixed asset types) — continuation 11

Exercise 5️⃣ Grant access with a real Access Contract (mixed asset types) — continuation 11 and compare the actual response/evidence with the expected feature behavior.

**Input:** `use_access_contract_kv`, `access_contract_kv_name`, `name`, `granted_apis`, `gateway_url`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('5️⃣ Grant access with a real Access Contract (mixed asset types) — continuation 11', inputs={'use_access_contract_kv': globals().get('use_access_contract_kv'), 'access_contract_kv_name': globals().get('access_contract_kv_name'), 'name': globals().get('name'), 'granted_apis': globals().get('granted_apis'), 'gateway_url': globals().get('gateway_url')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if use_access_contract_kv:
        from azure.identity import DefaultAzureCredential
        from azure.keyvault.secrets import SecretClient
        vault = SecretClient(f"https://{access_contract_kv_name}.vault.azure.net", DefaultAzureCredential())
        key_ok = bool(access_contract_key_secret_name) and vault.get_secret(access_contract_key_secret_name).value == api_key
        endpoint_values = [vault.get_secret(name).value for name in access_contract_endpoint_secret_names]
        run.check("access-contract-kv-secrets", key_ok and len(endpoint_values) == len(granted_apis)
            and all(value and value.startswith(gateway_url) for value in endpoint_values),
            expected="Shared key matches exact subscription; one gateway endpoint per granted API",
            actual=len(endpoint_values))
    else:
        run.skip("access-contract-kv-secrets", "Key Vault publishing disabled explicitly.")


<a id='6'></a>
### 6️⃣ Validate the MCP tools (handshake + tools/list)

Perform the MCP `initialize` handshake and list tools over the streamable HTTP transport for both published tool servers.

> **Endpoint convention:** with `useAssetTypePathPrefix = true` (default) every **tool** is served under an `mcp/` prefix and every **agent** under an `agent/` prefix. For an **API→MCP** tool (`mcp-from-api`) APIM also appends `/mcp`, so the endpoint is `{gateway}/mcp/{path}/mcp`. For a **native/remote MCP** server (`mcp-existing`) the endpoint is `{gateway}/mcp/{path}` (no trailing `/mcp`). The cells below prefer the authoritative `endpoint` returned by the deployment (`publishedAssets`) and fall back to computing the prefixed path. Set the toggle to `false` (or a per-asset `pathPrefix: ''`) to keep legacy un-prefixed paths.


### 6️⃣ Validate the MCP tools (handshake + tools/list)

Exercise 6️⃣ Validate the MCP tools (handshake + tools/list) and compare the actual response/evidence with the expected feature behavior.

**Input:** `assets`, `r`, `asset`, `gateway_url`, `a`, `results`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('6️⃣ Validate the MCP tools (handshake + tools/list)', inputs={'assets': globals().get('assets'), 'r': globals().get('r'), 'asset': globals().get('asset'), 'gateway_url': globals().get('gateway_url'), 'a': globals().get('a'), 'results': globals().get('results')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    def mcp_call(endpoint, api_key, method, params=None, mcp_session=None):
        headers = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream", "api-key": api_key}
        if mcp_session:
            headers["Mcp-Session-Id"] = mcp_session
        # APIM's MCP runtime requires a NUMERIC JSON-RPC id; a non-numeric string (e.g. a UUID) is rejected as 'Invalid JSON payload'.
        mcp_call._id = getattr(mcp_call, "_id", 0) + 1
        payload = {"jsonrpc": "2.0", "id": mcp_call._id, "method": method}
        if params is not None:
            payload["params"] = params
        r = utils.http_call(requests.post, endpoint, headers=headers, data=json.dumps(payload), timeout=60)
        text, data = r.text, None
        if "text/event-stream" in r.headers.get("content-type", ""):
            for line in text.splitlines():
                if line.startswith("data:"):
                    try:
                        data = json.loads(line[5:].strip()); break
                    except Exception:
                        pass
        else:
            try:
                data = r.json()
            except Exception:
                pass

        if not isinstance(data, dict) or data.get("jsonrpc") != "2.0" or data.get("id") != payload["id"]:
            data = {"error": {"code": "invalid_json_rpc", "message": "Malformed or mismatched JSON-RPC response"}}
        return r, data


    def validate_mcp(endpoint, api_key, label):
        try:
            initial, data = mcp_call(endpoint, api_key, "initialize", {
                "protocolVersion": "2025-06-18", "capabilities": {},
                "clientInfo": {"name": "citadel-validation", "version": "1.0"}})
            result = (data or {}).get("result", {})
            handshake_ok = initial.status_code == 200 and bool(
                result.get("protocolVersion") and result.get("serverInfo") and "capabilities" in result)
            run.check(label + "/initialize", handshake_ok and not (data or {}).get("error"),
                expected="JSON-RPC initialize result/protocolVersion/serverInfo/capabilities", actual=initial.status_code)
            session = initial.headers.get("Mcp-Session-Id")
            notification_headers = {"api-key": api_key, "Content-Type": "application/json", "Accept": "application/json, text/event-stream"}
            if session:
                notification_headers["Mcp-Session-Id"] = session
            utils.http_call(requests.post, endpoint, headers=notification_headers,
                json={"jsonrpc": "2.0", "method": "notifications/initialized"}, timeout=60)
            listed, data = mcp_call(endpoint, api_key, "tools/list", mcp_session=session)
            tools = (data or {}).get("result", {}).get("tools", [])
            tools_ok = listed.status_code == 200 and bool(tools) and all(
                tool.get("name") and isinstance(tool.get("inputSchema"), dict) for tool in tools)
            run.check(label + "/tools-list", tools_ok and not (data or {}).get("error"),
                expected="Nonempty named tools with inputSchema", actual=listed.status_code)
            return handshake_ok and tools_ok
        except Exception as exc:
            run.check(label + "/mcp", False, expected="MCP handshake/list", actual=str(exc))
            return False

    _ASSET_TYPE_PREFIX = {"mcp-from-api": "mcp", "mcp-existing": "mcp", "a2a": "agent"}

    def _prefixed_path(asset):
        prefix = _ASSET_TYPE_PREFIX.get(asset["assetType"], "")
        return f"{prefix}/{asset['path']}" if prefix else asset["path"]

    def mcp_endpoint(asset):
        # Prefer the authoritative endpoint returned by the deployment; fall back to the prefixed path.
        dep = globals().get("published_by_name", {}).get(asset["name"])
        if dep and dep.get("endpoint"):
            return dep["endpoint"]
        base = f"{gateway_url}/{_prefixed_path(asset)}"
        return f"{base}/mcp" if asset["assetType"] == "mcp-from-api" else base

    def agent_endpoint(asset):
        # Prefer the deployed (prefixed) path; fall back to the mirrored agent/ prefix.
        dep = globals().get("published_by_name", {}).get(asset["name"])
        if dep and dep.get("path"):
            return f"{gateway_url}/{dep['path']}"
        return f"{gateway_url}/{_prefixed_path(asset)}"

    for a in assets:
        if a['assetType'] in ('mcp-from-api', 'mcp-existing'):
            results[a['name']] = validate_mcp(mcp_endpoint(a), api_key, a['displayName'])


<a id='7'></a>
### 7️⃣ Validate the A2A agent (agent card + JSON-RPC)

Fetch the agent card that APIM re-exposes at `/.well-known/agent.json`, then send a JSON-RPC `message/send` that the gateway proxies (with its managed identity) to the Foundry agent.

### 7️⃣ Validate the A2A agent (agent card + JSON-RPC)

Run the actual 7️⃣ Validate the A2A agent (agent card + JSON-RPC) SDK example against the initialized contract and retain its returned content/session evidence.

**Input:** `enable_a2a_asset`, `a`, `body`, `assets`

**Output:** SDK responses, session/agent state, and recorded feature validation; optional disabled paths remain explicit.


In [ ]:
with utils.notebook_step('7️⃣ Validate the A2A agent (agent card + JSON-RPC)', inputs={'enable_a2a_asset': globals().get('enable_a2a_asset'), 'a': globals().get('a'), 'body': globals().get('body'), 'assets': globals().get('assets')}, expected='SDK responses, session/agent state, and recorded feature validation; optional disabled paths remain explicit.', namespace=globals()):
    if enable_a2a_asset:
        hr_agent_asset = next(a for a in assets if a["assetType"] == "a2a")
        agent_base = agent_endpoint(hr_agent_asset)
        card_url = f"{agent_base}/.well-known/agent.json"
        card_response = transport.request("GET", card_url, headers={"api-key": api_key})
        card = card_response.json_body if isinstance(card_response.json_body, dict) else {}
        card_url_value = card.get("url") or next((i.get("url") for i in card.get("supportedInterfaces", []) if i.get("url")), "")
        run.check("a2a-card", card_response.status_code == 200 and all(
            card.get(k) for k in ("name", "version", "skills")) and card_url_value.startswith(gateway_url),
            expected="Named/versioned/skilled agent card advertising gateway transport", actual=card_response.status_code)
        rpc = {"jsonrpc": "2.0", "id": 1, "method": "message/send",
            "params": {"message": {"kind": "message", "role": "user", "messageId": str(uuid.uuid4()),
                                  "parts": [{"kind": "text", "text": "What can you help me with?"}]}}}
        reply = transport.request("POST", agent_base, headers={"api-key": api_key}, json=rpc)
        body = reply.json_body if isinstance(reply.json_body, dict) else {}
        result = body.get("result", {})
        run.check("a2a-message-send", reply.status_code == 200 and body.get("jsonrpc") == "2.0"
            and body.get("id") == 1 and not body.get("error") and bool(result)
            and result.get("status", {}).get("state") not in ("failed", "rejected", "canceled")
            and (result.get("kind") in ("message", "task") or bool(result.get("parts") or result.get("artifacts"))),
            expected="Successful JSON-RPC message/task result", actual=reply.status_code)
    else:
        run.skip("a2a-card-and-invocation", "A2A asset explicitly disabled.")


<a id='8'></a>
### 8️⃣ Validate usage tracking (custom metrics)

The baseline policies emit `McpRequests` (namespace `mcp-usage`) and `A2ARequests` (namespace `a2a-usage`) on each inbound call. Query Application Insights `customMetrics` to confirm they land (metrics may take a couple of minutes to appear).

### 8️⃣ Validate usage tracking (custom metrics)

Query or inspect 8️⃣ Validate usage tracking (custom metrics) evidence for this run; configuration and HTTP success alone are not evidence.

**Input:** `app_insights_name_override`, `telemetry_poll_attempts`, `governance_hub_resource_group`, `enable_a2a_asset`, `run_started_at`

**Output:** Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.


In [ ]:
with utils.notebook_step('8️⃣ Validate usage tracking (custom metrics)', inputs={'app_insights_name_override': globals().get('app_insights_name_override'), 'telemetry_poll_attempts': globals().get('telemetry_poll_attempts'), 'governance_hub_resource_group': globals().get('governance_hub_resource_group'), 'enable_a2a_asset': globals().get('enable_a2a_asset'), 'run_started_at': globals().get('run_started_at')}, expected='Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.', namespace=globals()):
    app_insights_name = app_insights_name_override
    if not app_insights_name:
        run.blocked("published-usage-metrics", "Select app_insights_name_override; missing/ambiguous telemetry resource.")
    else:
        expected_metrics = {"McpRequests"} | ({"A2ARequests"} if enable_a2a_asset else set())
        asset_names = ",".join("'" + name + "'" for name in asset_metric_names)
        kql = (f"customMetrics | where timestamp >= datetime({run_started_at}) "
               f"| where tostring(customDimensions['appId']) == '{sub_name}' "
               f"| where tostring(customDimensions['deploymentName']) in ({asset_names}) "
               "| where name in ('McpRequests','A2ARequests') "
               "| summarize observations=sum(valueSum) by name")
        observed = set()
        for attempt in range(telemetry_poll_attempts):
            query = utils.run(f'az monitor app-insights query --app {app_insights_name} --subscription {app_insights_subscription_id} -g {app_insights_resource_group} --analytics-query "{kql}" -o json')
            tables = query.json_data.get("tables", []) if query.success and isinstance(query.json_data, dict) else []
            rows = tables[0].get("rows", []) if tables else []
            observed = {row[0] for row in rows if len(row) > 1 and row[1] > 0}
            if expected_metrics <= observed:
                break
            if attempt + 1 < telemetry_poll_attempts:
                time.sleep(telemetry_poll_seconds)
        if not expected_metrics <= observed:
            run.blocked("published-usage-metrics", f"No fresh metric evidence for {sorted(expected_metrics - observed)} within bounded polling window.")
        else:
            run.check("published-usage-metrics", True, expected=sorted(expected_metrics), actual=sorted(observed))


<a id='9'></a>
### 9️⃣ Validate resiliency (circuit breaker on published backends)

Confirm the remote MCP and A2A assets got a native circuit breaker on their backend.

### 9️⃣ Validate resiliency (circuit breaker on published backends)

Exercise 9️⃣ Validate resiliency (circuit breaker on published backends) and compare the actual response/evidence with the expected feature behavior.

**Input:** `a`, `assets`, `name`, `backend_id`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('9️⃣ Validate resiliency (circuit breaker on published backends)', inputs={'a': globals().get('a'), 'assets': globals().get('assets'), 'name': globals().get('name'), 'backend_id': globals().get('backend_id')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    for name in [a['name'] for a in assets if a['assetType'] in ('mcp-existing', 'a2a')]:
        backend_id = f"{name}-backend"
        uri = f"{apimClientTool.apim_service_id}/backends/{backend_id}?api-version=2024-06-01-preview"
        b = utils.run(f"az rest --method get --uri {uri} -o json")
        if b.success and b.json_data:
            cb = b.json_data.get('properties', {}).get('circuitBreaker')
            if cb and cb.get('rules'):
                rule = cb['rules'][0]
                fc = rule.get('failureCondition', {})
                utils.print_ok(f"{backend_id}: circuit breaker ON (count={fc.get('count')}, interval={fc.get('interval')}, trip={rule.get('tripDuration')})")
            else:
                utils.print_warning(f"{backend_id}: no circuit breaker configured")
        else:
            utils.print_error(f"{backend_id}: backend not found")
        run.check("circuit-breaker-config/" + name,
            b.success and bool(b.json_data and b.json_data.get("properties", {}).get("circuitBreaker", {}).get("rules")),
            expected="Configured native circuit-breaker rules (not trip/recovery)", actual=b.success)
    run.skip("circuit-breaker-trip-recovery", "Disruptive fault injection is not executed; only configuration is validated.")


<a id='10'></a>
### 🔟 Consume the published assets (Microsoft Agent Framework + direct MCP)

Prove the published assets are usable by a real client, **through the gateway**:
1. A **Microsoft Agent Framework** `A2AAgent` resolves the published **HR agent** card and answers an HR question.
2. A **direct MCP `tools/call`** invokes a published tool through the gateway.

> The publish contract **rewrites the agent card's transport URLs to the gateway**, so the A2A client routes **through the gateway** (presenting the subscription `api-key`) instead of calling Foundry directly. The compatible Agent Framework core/A2A packages and `a2a-sdk` are installed together by [Validation Environment Setup](citadel-environment-setup.ipynb). Select that same `.venv-validation-local` kernel; do not upgrade these packages individually.

### 🔟 Consume the published assets (Microsoft Agent Framework + direct MCP)

Run the actual 🔟 Consume the published assets (Microsoft Agent Framework + direct MCP) SDK example against the initialized contract and retain its returned content/session evidence.

**Input:** `enable_agent_framework`, `enable_a2a_asset`, `a2a_card_backend_url`, `results`, `a`

**Output:** SDK responses, session/agent state, and recorded feature validation; optional disabled paths remain explicit.


In [ ]:
with utils.notebook_step('🔟 Consume the published assets (Microsoft Agent Framework + direct MCP)', inputs={'enable_agent_framework': globals().get('enable_agent_framework'), 'enable_a2a_asset': globals().get('enable_a2a_asset'), 'a2a_card_backend_url': globals().get('a2a_card_backend_url'), 'results': globals().get('results'), 'a': globals().get('a')}, expected='SDK responses, session/agent state, and recorded feature validation; optional disabled paths remain explicit.', namespace=globals()):
    # Microsoft Agent Framework: consume the published HR agent (A2A) through the gateway.
    # The published agent card advertises the gateway's transport URLs (rewritten by the publish contract),
    # so A2AAgent routes through the gateway (presenting the subscription api-key) rather than calling Foundry directly.
    if enable_agent_framework:
        import nest_asyncio, asyncio, httpx
        from a2a.client import A2ACardResolver
        from agent_framework.a2a import A2AAgent
        nest_asyncio.apply()

    if enable_agent_framework and enable_a2a_asset and a2a_card_backend_url:
        agent_url = agent_endpoint(next(a for a in assets if a["assetType"] == "a2a"))

        async def ask_hr_agent(question):
            # A2A API is subscriptionRequired; present the test api-key on every gateway call (card + JSON-RPC).
            async with httpx.AsyncClient(timeout=120.0, headers={"api-key": api_key}) as http_client:
                resolver = A2ACardResolver(httpx_client=http_client, base_url=agent_url)
                card = await resolver.get_agent_card(relative_card_path="/.well-known/agent.json")
                agent = A2AAgent(name=card.name, description=card.description, agent_card=card, http_client=http_client)
                resp = await agent.run(question)
                return "\n".join(getattr(m, "text", "") for m in resp.messages)

        try:
            utils.print_info("Asking the HR agent (Microsoft Agent Framework -> A2A -> gateway)...")
            answer = asyncio.run(ask_hr_agent("What is our leave policy?"))
            utils.print_ok("HR agent answered:")
            utils.format_output('🔟 Consume the published assets (Microsoft Agent Framework + direct MCP)', {'answer': answer}, status='info', part='RESULT')
            results['hr-chat-agent-maf'] = bool(answer.strip())
        except Exception as e:
            utils.print_error(f"Agent Framework A2A invocation failed: {e}")
            utils.print_warning("Verify the shared environment with citadel-environment-setup.ipynb, then restart "
                                "and select the same .venv-validation-local kernel; do not upgrade A2A packages individually.")
            results['hr-chat-agent-maf'] = False
    else:
        utils.print_warning("A2A asset not published; skipping Agent Framework A2A consumption.")
    if enable_agent_framework and enable_a2a_asset:
        run.check("agent-framework-a2a", results.get("hr-chat-agent-maf", False),
            expected="Nonempty SDK agent answer via gateway", actual=results.get("hr-chat-agent-maf"))
    else:
        run.skip("agent-framework-a2a", "Optional SDK integration/A2A disabled explicitly.")


### 🔟 Consume the published assets (Microsoft Agent Framework + direct MCP) — continuation 17

Run the actual 🔟 Consume the published assets (Microsoft Agent Framework + direct MCP) — continuation 17 SDK example against the initialized contract and retain its returned content/session evidence.

**Input:** `session`, `asset`, `data`, `a`, `text`, `results`

**Output:** SDK responses, session/agent state, and recorded feature validation; optional disabled paths remain explicit.


In [ ]:
with utils.notebook_step('🔟 Consume the published assets (Microsoft Agent Framework + direct MCP) — continuation 17', inputs={'session': globals().get('session'), 'asset': globals().get('asset'), 'data': globals().get('data'), 'a': globals().get('a'), 'text': globals().get('text'), 'results': globals().get('results')}, expected='SDK responses, session/agent state, and recorded feature validation; optional disabled paths remain explicit.', namespace=globals()):
    # Direct invocation of a published MCP tool (initialize -> tools/call) through the gateway.
    def call_mcp_tool(asset, tool_name, arguments):
        ep = mcp_endpoint(asset)
        r, _ = mcp_call(ep, api_key, "initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                        "clientInfo": {"name": "citadel-validation", "version": "1.0"}})
        session = r.headers.get("Mcp-Session-Id")

        notification_headers = {"api-key": api_key, "Content-Type": "application/json", "Accept": "application/json, text/event-stream"}
        if session:
            notification_headers["Mcp-Session-Id"] = session
        utils.http_call(requests.post, ep, headers=notification_headers,
            json={"jsonrpc": "2.0", "method": "notifications/initialized"}, timeout=60)
        r, data = mcp_call(ep, api_key, "tools/call", {"name": tool_name, "arguments": arguments}, mcp_session=session)
        return data if r.status_code == 200 else None

    weather_asset = next((a for a in assets if a["name"] == "weather-tool"), None)
    if weather_asset:
        out = call_mcp_tool(weather_asset, "get-weather", {"city": "London"})
        content = (out or {}).get("result", {}).get("content", [])
        text = content[0].get("text", "") if content else json.dumps(out)
        utils.print_ok("Direct MCP tools/call  weather-tool.get-weather('London') ->")
        utils.format_output('🔟 Consume the published assets (Microsoft Agent Framework + direct MCP) — continuation 17', {'text': text}, status='info', part='RESULT')
        results['weather-tool-toolcall'] = bool(content)
    else:
        utils.print_warning("weather-tool not in assets; skipping direct MCP tool invocation.")
    run.check("weather-tools-call", results.get("weather-tool-toolcall", False)
        and not (out or {}).get("error") and not (out or {}).get("result", {}).get("isError", False),
        expected="Successful tools/call with nonempty content, no JSON-RPC/tool error",
        actual=results.get("weather-tool-toolcall"))


<a id='11'></a>
### 1️⃣1️⃣ Validate conditional access-contract policies (per asset type)

The `MULTI-` product applies **different** throttling per asset type via the `set-asset-kind` branch. Burst-call a **Tool** and the **Agent** to confirm the request-based `rate-limit-by-key` trips (**HTTP 429**) — proving the product policy branched on asset kind and applied the Tool/Agent controls (not the LLM token limit). LLM token limits are validated best-effort only when a model is reachable.

### 1️⃣1️⃣ Validate conditional access-contract policies (per asset type)

Exercise 1️⃣1️⃣ Validate conditional access-contract policies (per asset type) and compare the actual response/evidence with the expected feature behavior.

**Input:** `enable_a2a_asset`, `a2a_card_backend_url`, `a`, `ep`, `body`, `results`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('1️⃣1️⃣ Validate conditional access-contract policies (per asset type)', inputs={'enable_a2a_asset': globals().get('enable_a2a_asset'), 'a2a_card_backend_url': globals().get('a2a_card_backend_url'), 'a': globals().get('a'), 'ep': globals().get('ep'), 'body': globals().get('body'), 'results': globals().get('results')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    import concurrent.futures

    def _burst(url, headers, body, n):
        def _one(_):
            try:
                response = utils.http_call(requests.post, url, headers=headers, data=body, timeout=30)
                return (response.status_code, response.text, response.headers)
            except Exception as exc:
                return (None, str(exc), {})
        with concurrent.futures.ThreadPoolExecutor(max_workers=10) as ex:
            return list(ex.map(_one, range(n)))

    # --- Tool: burst past the tool rate limit -> expect 429s (throttled calls never reach the backend) ---
    tool_asset = next((a for a in assets if a["assetType"] in ("mcp-from-api", "mcp-existing")), None)
    if tool_asset:
        ep = mcp_endpoint(tool_asset)
        hdr = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream", "api-key": api_key}
        body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": "initialize",
                           "params": {"protocolVersion": "2025-06-18", "capabilities": {},
                                      "clientInfo": {"name": "burst", "version": "1.0"}}})
        codes = _burst(ep, hdr, body, TOOL_CALLS_PER_MIN + 15)
        throttled = sum(1 for c in codes if c[0] == 429 and (c[2].get("Retry-After") or "rate" in c[1].lower()))
        (utils.print_ok if throttled else utils.print_warning)(
            f"Tool rate-limit: {throttled}/{len(codes)} calls got 429 (limit {TOOL_CALLS_PER_MIN}/min) -> tool branch applied")
        results['tool-rate-limit-429'] = throttled > 0
    else:
        utils.print_warning("No tool asset; skipping tool rate-limit check.")

    # --- Agent: burst past the agent rate limit -> expect 429s ---
    if enable_a2a_asset and a2a_card_backend_url:
        agent_base = agent_endpoint(next(a for a in assets if a["assetType"] == "a2a"))
        hdr = {"Content-Type": "application/json", "api-key": api_key, "A2A-Version": "1.0"}
        body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": "message/send",
                           "params": {"message": {"kind": "message", "role": "user", "messageId": "burst",
                                                  "parts": [{"kind": "text", "text": "ping"}]}}})
        codes = _burst(agent_base, hdr, body, AGENT_CALLS_PER_MIN + 15)
        throttled = sum(1 for c in codes if c[0] == 429 and (c[2].get("Retry-After") or "rate" in c[1].lower()))
        (utils.print_ok if throttled else utils.print_warning)(
            f"Agent rate-limit: {throttled}/{len(codes)} calls got 429 (limit {AGENT_CALLS_PER_MIN}/min) -> agent branch applied")
        results['agent-rate-limit-429'] = throttled > 0
    else:
        utils.print_warning("A2A asset not published; skipping agent rate-limit check.")
    run.check("tool-rate-limit", results.get("tool-rate-limit-429", False),
        expected="429 with rate-limit policy evidence", actual=results.get("tool-rate-limit-429"))
    if enable_a2a_asset:
        run.check("agent-rate-limit", results.get("agent-rate-limit-429", False),
            expected="429 with rate-limit policy evidence", actual=results.get("agent-rate-limit-429"))
    else:
        run.skip("agent-rate-limit", "A2A asset disabled explicitly.")


<a id='results'></a>
### 📊 Results Summary

### 📊 Results Summary

Collect the final 📊 Results Summary verdict without discarding earlier failed or blocked required cases.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** The strict validation summary; any failed or blocked required coverage raises.


In [ ]:
with utils.notebook_step('📊 Results Summary', inputs={}, expected='The strict validation summary; any failed or blocked required coverage raises.', namespace=globals()):
    run.finish()


<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

Remove the temporary test product/subscription. Set `delete_published_assets = True` to also remove the published APIs/backends.

### 🧹 Cleanup (Optional)

Apply the explicitly enabled 🧹 Cleanup (Optional) action only to resources owned by this run.

**Input:** `owned_products`, `cleanup_delete_contract`, `owned_asset_apis`, `owned_asset_backends`, `rg`

**Output:** Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.


In [ ]:
with utils.notebook_step('🧹 Cleanup (Optional)', inputs={'owned_products': globals().get('owned_products'), 'cleanup_delete_contract': globals().get('cleanup_delete_contract'), 'owned_asset_apis': globals().get('owned_asset_apis'), 'owned_asset_backends': globals().get('owned_asset_backends'), 'rg': globals().get('rg')}, expected='Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.', namespace=globals()):
    delete_published_assets = False
    cleanup_products(apimClientTool, owned_products, enabled=cleanup_delete_contract)
    utils.format_output('🧹 Cleanup (Optional)', {'value_1': "Owned asset APIs:", 'value_2': sorted(owned_asset_apis), 'value_3': "owned backends:", 'value_4': sorted(owned_asset_backends)}, status='info', part='RESULT')
    if delete_published_assets:
        for name in sorted(owned_asset_apis):
            client.api.delete(rg, svc, name, if_match="*")
        for name in sorted(owned_asset_backends):
            client.backend.delete(rg, svc, name, if_match="*")
    # Reused APIs, the shared weather source, role grants and predecessor assets are never deleted.
